## Bounded Knapsack Problem
### Consession Stand
The manager has a rolling cooler for a sports event with a strict weight capacity. They want to load it with a combination of pre-packed bulk items to maximise their total profit. However, the stadium warehouse only has a limited stock of each item category available.

1. <b>Objective Function</b>: Maximize the profit generated from the selected concession items:
$$\text{maximize} \; Z = \sum_{j=1}^n p_j x_j$$ 
2. <b>Constraints</b>
* Cooler Weight Capacity: Ensures that the total weight of all loaded concession items does not exceed the cooler's total weight limit
$$\sum_{j=1}^n w_jx_j \leq c$$ 
* Warehouse Inventory LWimit: The number of items loaded into the cooler cannot exceed the available quantity in the warehouse. 
$$x_j \le b_j \quad \forall j$$
* Integrity and Non-Negativity: Concession item quantities cannot be negative or fractional.
$$ x_j \in \mathbb{Z}_{\geq0} \quad \forall j$$
3. <b>Decision Variable</b>
* $x_j$: represents the number of units of concession item $j$ to be loaded into the rolling cooler. 
4. <b>Parameters</b>
* $n$: the total number of distinct item types available, indexed by $j$
* $p_j$: the profit associated with a single unit of item type $j$
* $w_j$: the weight of a single unit of item type $j$
* $b_j$: the maximum available quantity of item type $j$
* $c$: the total capacity of the cooler

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>concession_stand</code>.

In [88]:
from docplex.mp.model import Model

mdl = Model(name="concession_stand")

#### Read the CSV file containing the input data.
Read the CSV file containing the input data.

In [89]:
import pandas as pd
concession_df = pd.read_csv("concession_stand_data.csv", encoding="utf-8")

In [90]:
concession_df

,item_id,item_name,profit_per_unit,weight_per_unit,warehouse_stock
0,1,Bottled Water,2.5,1.1,30
1,2,Cola Cans,3.0,0.9,40
2,3,Hot Dogs (Bulk Pack),5.5,2.5,15
3,4,Popcorn Bags,4.0,0.5,25
4,5,Soft Pretzels,3.5,1.2,20
5,6,Energy Drinks,4.5,0.8,25


#### Define Parameters 
Extract the following parameters from the columns of the concession dataframe:
* the profit $p$ per unit of item $j$
* the weight $w$ per unit of item $j$
* the warehouse stock $b$ of item $j$

We also extracted the <code>item_id</code>s and the <code>item_name</code>s of the items from the dataframe. Then, we set the maximum capacity $c$ of the cooler to 60.

In [92]:
items = concession_df["item_id"].tolist()
names = dict(zip(concession_df["item_id"], concession_df["item_name"]))
p = dict(zip(concession_df["item_id"], concession_df["profit_per_unit"]))
w = dict(zip(concession_df["item_id"], concession_df["weight_per_unit"]))
b = dict(zip(concession_df["item_id"], concession_df["warehouse_stock"]))
c = 60

#### Define the Decision Variables
Integer variables for the items

In [93]:
x = mdl.integer_var_dict(items, lb=0, name="quantity")

#### Define the Constraints
Cooler Weight Capacity

In [94]:
mdl.add_constraint(mdl.sum(w[j] * x[j] for j in items) <= c, ctname="Cooler_Weight_Capacity")

docplex.mp.LinearConstraint[Cooler_Weight_Capacity](1.100quantity_1+0.900quantity_2+2.500quantity_3+0.500quantity_4+1.200quantity_5+0.800quantity_6,LE,60)

Warehouse Inventory Limit

In [95]:
for j in items:
    mdl.add_constraint(x[j] <= b[j], ctname=f"Weight_Limit_{j}")

#### Define the Objective Function

In [96]:
mdl.maximize(mdl.sum(p[j] * x[j] for j in items))

#### Solve the Model

In [97]:
print("Solving model...")
solution = mdl.solve(log_output=True)

Solving model...
Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 0.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 6 rows and 0 columns.
MIP Presolve added 2 rows and 2 columns.
MIP Presolve modified 1 coefficients.
Reduced MIP has 3 rows, 8 columns, and 12 nonzeros.
Reduced MIP has 0 binaries, 8 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 2 rows and 2 columns.
MIP Presolve added 2 rows and 2 columns.
Reduced MIP has 3 rows, 8 columns, and 12 nonzeros.
Reduced MIP has 0 binaries, 8 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 10 threads.
Root relaxation solution time = 0.00 sec. (0.01 ticks)

        Nodes                               

#### Print the Solution

In [109]:
item_names = []
quantities = []
weights = []

if solution:
    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Status: {mdl.get_solve_status()}")
    print(f"Total Profit: ${solution.objective_value:.0f}")
    total_weight = 0
    
    for j in items:
        qty = solution.get_value(x[j])
        if qty > 0:
            item_names.append(concession_df.loc[j-1,'item_name'])
            quantities.append(qty)
            item_weight = qty * w[j]
            weights.append(item_weight)
            total_weight += item_weight
    df = pd.DataFrame({'Item': item_names, 'Quantity': quantities, 'Weight (in lbs)': weights})
    print(df)
    print(f"Total Cooler Weight: {total_weight:.1f} / {c} lbs")
else:
    print("Could not find a feasible packing plan.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Profit: $303
            Item  Quantity  Weight (in lbs)
0      Cola Cans      29.0             26.1
1   Popcorn Bags      25.0             12.5
2  Soft Pretzels       1.0              1.2
3  Energy Drinks      25.0             20.0
Total Cooler Weight: 59.8 / 60 lbs


In [110]:
df

,Item,Quantity,Weight (in lbs)
0,Cola Cans,29.0,26.1
1,Popcorn Bags,25.0,12.5
2,Soft Pretzels,1.0,1.2
3,Energy Drinks,25.0,20.0


## Multiple Knapsack Problem
### Faculty Assignment
Faculty assignment refers to the assignment of professors to teach specific university courses. This can be modelled as a Generalised Assignment Program and of course can be thought of as a Multiple Knapsack with Assignment Restrictions
1. <b>Objective Function</b>: Maximize the total faculty specialisation/preference
$$\text{maximize} \; Z = \sum_{i=1}^m \sum_{j=1}^n p_{ij}x_j$$ 
2. <b>Constraints</b>
* Workload Constraint: Each professor has a load limit.
$$\sum_{j=1}^m w_{j}x_{ij} \leq L_i \quad \forall i$$
* Coverage Constraint: Each course must be assigned to a professor.
$$ \sum_{i=1}^n x_{ij} = 1 \quad \forall j$$
* Minimum Assignment Constraint: Every professor must be assigned to at least 1 course.
$$ \sum_{j=1}^m x_{ij} \geq 1 \quad \forall i$$
3. <b>Decision Variables</b>:
* $x_{ij} \in \{0,1\}$: 1 if professor $i$ is assigned course $j$, and 0 if it is skipped.
4. <b>Parameters</b>
* $w_j$: the weight (teaching units) of course $j$
* $L_j$: the maximum capacity (maximum load units) of professor $i$ 
* $p_{ij}$: the preference score of assigning professor $i$ to course $j$ 

#### Define the model name
Import `Model` from `docplex.mp.model` and create a `Model` object named <code>faculty_loading</code.

In [57]:
from docplex.mp.model import Model

mdl = Model(name="faculty_loading")

#### Read the CSV file containing the input data.
Read the two CSV files containing the faculty input and the courses input.

In [71]:
import pandas as pd
faculty_df = pd.read_csv("faculty_loading_data2.csv", encoding="utf-8")
courses_df = pd.read_csv("courses_data2.csv", encoding="utf-8")

In [72]:
faculty_df

,ID,Faculty,CS101_1A,CS101_1B,CS201_2A,CS201_2B,CS301_3A,CS301_3B,CS401_4B,CS401_4A,CS402_4B,CS402_4A,Max_Load
0,1,Prof_Bulao,8,8,6,6,10,10,9,9,4,4,9
1,2,Prof_Geralde,5,5,5,5,7,7,8,8,10,10,18
2,3,Prof_Alino,10,10,9,9,4,4,2,2,5,5,15
3,4,Prof_Tacadao,3,3,5,5,7,7,10,10,10,10,18
4,5,Prof_Nebrao,5,5,8,8,2,2,10,10,5,5,18


In [73]:
courses_df

,Course,Credit
0,CS101_1A,3
1,CS101_1B,3
2,CS201_2A,3
3,CS201_2B,3
4,CS301_3A,3
5,CS301_3B,3
6,CS401_4A,3
7,CS401_4B,3
8,CS402_4A,3
9,CS402_4B,3


#### Define the parameters
1. To get the preferences (profit) $p$,
<ol style="list-style-type: lower-alpha;">
<li>Get the course list from the <code>Course</code> column of the courses dataframe.</li>
<li>Set the <code>Faculty</code> column as index, then get the course preferences by name using the course list. Save this into the preferences dataframe.</li>
<li>Extract the faculty list from the index.</li>
<li>Flatten the profit dataframe into a 2d dictionary. </li>
</ol>
2. Get the course credits $w$ from the <code>Credit</code> column in the courses dataframe by setting the index to the <code>Course</code> column and then convert to a dictionary.
3. Get the maximum load of the professors from the <code>Max_Load</code> column of the faculty dataframe by setting the index to the <code>Faculty</code> column and then convert this again to a dictionary. 

In [74]:
courses = courses_df['Course'].tolist()
p_df = faculty_df.set_index('Faculty')[courses]
faculty = p_df.index.to_list()
p = p_df.stack().to_dict()

In [75]:
p_df

,CS101_1A,CS101_1B,CS201_2A,CS201_2B,CS301_3A,CS301_3B,CS401_4A,CS401_4B,CS402_4A,CS402_4B
Faculty,,,,,,,,,,
Prof_Bulao,8,8,6,6,10,10,9,9,4,4
Prof_Geralde,5,5,5,5,7,7,8,8,10,10
Prof_Alino,10,10,9,9,4,4,2,2,5,5
Prof_Tacadao,3,3,5,5,7,7,10,10,10,10
Prof_Nebrao,5,5,8,8,2,2,10,10,5,5


In [76]:
w = courses_df.set_index('Course')['Credit'].to_dict()
w

{'CS101_1A': 3,
 'CS101_1B': 3,
 'CS201_2A': 3,
 'CS201_2B': 3,
 'CS301_3A': 3,
 'CS301_3B': 3,
 'CS401_4A': 3,
 'CS401_4B': 3,
 'CS402_4A': 3,
 'CS402_4B': 3}

In [77]:
L = faculty_df.set_index('Faculty')['Max_Load'].to_dict()
L

{'Prof_Bulao': 9,
 'Prof_Geralde': 18,
 'Prof_Alino': 15,
 'Prof_Tacadao': 18,
 'Prof_Nebrao': 18}

#### Define the Decision Variables
Binary variables for the faculty-course assignment

In [78]:
x = mdl.binary_var_dict([(i, j) for i in faculty for j in courses], name="x")

#### Define the Constraints
Workload Constraints

In [79]:
for i in faculty:
    mdl.add_constraint(
        mdl.sum(w[j] * x[(i, j)] for j in courses) <= L[i], 
        ctname=f"workload_limit_{i}"
    )

Coverage Constraints

In [80]:
for j in courses:
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for i in faculty) == 1, 
        ctname=f"course_coverage_{j}"
    )

Minimum Assignment Constraints

In [81]:
for i in faculty:
    mdl.add_constraint(
        mdl.sum(x[(i, j)] for j in courses) >= 1, 
        ctname=f"faculty_assignment_{i}"
    )

#### Define the Objective Function

In [82]:
total_preference = mdl.sum(p[(i, j)] * x[(i, j)] for i in faculty for j in courses)
mdl.maximize(total_preference)

#### Solve the Model

In [83]:
print("Solving model...")
solution = mdl.solve(log_output=True)

Solving model...
Version identifier: 22.1.1.0 | 2022-11-28 | 9160aff4d
CPXPARAM_Read_DataCheck                          1
1 of 3 MIP starts provided solutions.
MIP start 'm1' defined initial solution with objective 98.0000.
Tried aggregator 1 time.
MIP Presolve eliminated 4 rows and 0 columns.
MIP Presolve modified 6 coefficients.
Reduced MIP has 31 rows, 75 columns, and 205 nonzeros.
Reduced MIP has 75 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.12 ticks)
Probing time = 0.00 sec. (0.04 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 11 rows and 25 columns.
Reduced MIP has 20 rows, 50 columns, and 150 nonzeros.
Reduced MIP has 50 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.09 ticks)
Probing time = 0.00 sec. (0.03 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 20 rows, 50 columns, and 150 nonzeros.
Reduced MIP has 50 binaries, 0 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec.

#### Print the Solution

In [84]:
if solution:
    print("\n=== OPTIMAL SOLUTION FOUND ===")
    print(f"Status: {mdl.get_solve_status()}")
    print(f"Total Preferences: {solution.objective_value:,.2f}")
    for f in faculty:
            print(f"\nInstructor: {f}")
            assigned_any = False
            
            for c in courses:
                if x[(f, c)].solution_value > 0.5:
                    print(f"  • Is assigned {c} ")
                    assigned_any = True
                        
            if not assigned_any:
                print("  • No classes assigned.")
else:
    print("No feasible schedule found.")


=== OPTIMAL SOLUTION FOUND ===
Status: JobSolveStatus.OPTIMAL_SOLUTION
Total Preferences: 98.00

Instructor: Prof_Bulao
  • Is assigned CS301_3A 
  • Is assigned CS301_3B 

Instructor: Prof_Geralde
  • Is assigned CS402_4A 
  • Is assigned CS402_4B 

Instructor: Prof_Alino
  • Is assigned CS101_1A 
  • Is assigned CS101_1B 
  • Is assigned CS201_2A 
  • Is assigned CS201_2B 

Instructor: Prof_Tacadao
  • Is assigned CS401_4A 

Instructor: Prof_Nebrao
  • Is assigned CS401_4B 
